In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

Importing Libraries

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

In [ ]:
train = pd.read_csv('/kaggle/input/llm-classification-finetuning/train.csv')
test = pd.read_csv('/kaggle/input/llm-classification-finetuning/test.csv')
sample_submission = pd.read_csv('/kaggle/input/llm-classification-finetuning/sample_submission.csv')

Loading the Datasets

In [ ]:
print("Missing values in training data:")
print (train.isnull().sum())

Exploring the Data

In [ ]:
print("Missing values in test data:")
print (test.isnull().sum())

In [ ]:
print("Missing values in sample_submission:")
print (sample_submission.isnull().sum())

In [ ]:
from sklearn.preprocessing import LabelEncoder

In [ ]:
#Getting all the unique models for both train and test
all_models = list(set(train['model_a'].unique()) | set(train['model_b'].unique()))

Processing and Encoding

In [ ]:
model_encoder = LabelEncoder()

In [ ]:
#fit the encoder on all unique models so that model names can be transformed into a numeric value, ensuring that encoded values match
model_encoder.fit(all_models)

In [ ]:
#Encode the names in the training dataset
train['model_a_encoded'] = model_encoder.transform(train['model_a'])
train['model_b_encoded'] = model_encoder.transform(train['model_b'])

print(f"Encoded {len(all_models)} unique models from training data:")
for i, model in enumerate(model_encoder.classes_):
    print(f"  {i}: {model}")

In [ ]:
#Create text-based numeric features

def create_text_features(df):
    """ Create numeric features from text data"""
    
    # Text length features
    df['prompt_length'] = df['prompt'].str.len()
    df['response_a_length'] = df['response_a'].str.len()
    df['response_b_length'] = df['response_b'].str.len()
    
    # Word count features
    df['prompt_word_count'] = df['prompt'].str.split().str.len()
    df['response_a_word_count'] = df['response_a'].str.split().str.len()
    df['response_b_word_count'] = df['response_b'].str.split().str.len()
    
    # Character count features (excluding spaces)
    df['prompt_char_count'] = df['prompt'].str.replace(' ', '').str.len()
    df['response_a_char_count'] = df['response_a'].str.replace(' ', '').str.len()
    df['response_b_char_count'] = df['response_b'].str.replace(' ', '').str.len()
    
    # Average word length
    df['prompt_avg_word_length'] = df['prompt_char_count'] / (df['prompt_word_count'] + 1e-8)
    df['response_a_avg_word_length'] = df['response_a_char_count'] / (df['response_a_word_count'] + 1e-8)
    df['response_b_avg_word_length'] = df['response_b_char_count'] / (df['response_b_word_count'] + 1e-8)
    
    # Response length ratio (response length / prompt length)
    df['response_a_length_ratio'] = df['response_a_length'] / (df['prompt_length'] + 1e-8)
    df['response_b_length_ratio'] = df['response_b_length'] / (df['prompt_length'] + 1e-8)
    
    # Difference in response lengths
    df['response_length_diff'] = df['response_a_length'] - df['response_b_length']
    df['response_word_count_diff'] = df['response_a_word_count'] - df['response_b_word_count']
    
    # Punctuation counts
    import string
    df['prompt_punctuation_count'] = df['prompt'].str.count(f'[{string.punctuation}]')
    df['response_a_punctuation_count'] = df['response_a'].str.count(f'[{string.punctuation}]')
    df['response_b_punctuation_count'] = df['response_b'].str.count(f'[{string.punctuation}]')
    
    # Question marks and exclamation marks
    df['prompt_question_marks'] = df['prompt'].str.count('\?')
    df['response_a_question_marks'] = df['response_a'].str.count('\?')
    df['response_b_question_marks'] = df['response_b'].str.count('\?')
    
    df['prompt_exclamation_marks'] = df['prompt'].str.count('!')
    df['response_a_exclamation_marks'] = df['response_a'].str.count('!')
    df['response_b_exclamation_marks'] = df['response_b'].str.count('!')
    
    # Uppercase ratio
    df['prompt_uppercase_ratio'] = df['prompt'].str.count(r'[A-Z]') / (df['prompt_length'] + 1e-8)
    df['response_a_uppercase_ratio'] = df['response_a'].str.count(r'[A-Z]') / (df['response_a_length'] + 1e-8)
    df['response_b_uppercase_ratio'] = df['response_b'].str.count(r'[A-Z]') / (df['response_b_length'] + 1e-8)
    
    # Number count
    df['prompt_number_count'] = df['prompt'].str.count(r'\d')
    df['response_a_number_count'] = df['response_a'].str.count(r'\d')
    df['response_b_number_count'] = df['response_b'].str.count(r'\d')
    
    return df

In [ ]:
# Applying feature engineering to both datasets
print("Creating text-based features for training data...")
train = create_text_features(train)

print("Creating text-based features for test data...")
test = create_text_features(test)

print("\n" + "=" * 50)
print("FEATURE SUMMARY")
print("=" * 50)

In [ ]:
# Converting one-hot to numeric labels in one line
y = train[['winner_model_a', 'winner_model_b', 'winner_tie']].values.argmax(axis=1)

In [ ]:
# Droping target columns and other unnecessary columns from training data
columns_to_drop = [
    'id',                    # ID column - not useful for prediction
    'model_a', 'model_b',    # Original model names (I have encoded versions)
    'prompt', 'response_a', 'response_b',  # Raw text columns (I'll use TF-IDF later)
    'winner_model_a', 'winner_model_b', 'winner_tie'  # Target columns
]

# Creating feature matrix X by dropping unnecessary columns
X = train.drop(columns=columns_to_drop)

print(f"Feature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")
print(f"Features: {list(X.columns)}")

In [ ]:
#Now I will build the Model
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, log_loss

Building the Model

In [ ]:
# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    X, y, 
    test_size=0.2,           # 20% for validation
    random_state=42,         # For reproducibility
    stratify=y               # Maintain class distribution
)

print(f"Training set shape: {X_train.shape}")
print(f"Validation set shape: {X_val.shape}")
print(f"Training target distribution: {np.bincount(y_train)}")
print(f"Validation target distribution: {np.bincount(y_val)}")

In [ ]:
#Scaling the features, which is important for logistic regression

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

print("Features scaled successfully!")

In [ ]:
# Initializing LogisticRegression with appropriate parameters
model = LogisticRegression(
    random_state=42,         # For reproducibility
    max_iter=1000,          # Increase iterations for convergence
    multi_class='ovr',       # One-vs-Rest for multi-class
    solver='liblinear',      # Good for small datasets
    C=1.0                    # Regularization strength (1.0 = no regularization)
)

print("LogisticRegression model initialized!")
print(f"Model parameters: {model.get_params()}")

In [ ]:
# Training and fitting the model
print("Training the model...")
model.fit(X_train_scaled, y_train)
print("Model training completed!")

In [ ]:
# Lets Make predictions
y_train_pred = model.predict(X_train_scaled)
y_val_pred = model.predict(X_val_scaled)

In [ ]:
# Getting prediction probabilities for log loss
y_train_proba = model.predict_proba(X_train_scaled)
y_val_proba = model.predict_proba(X_val_scaled)

In [ ]:
# Calculating the log loss (the evaluation metric)
train_log_loss = log_loss(y_train, y_train_proba)
val_log_loss = log_loss(y_val, y_val_proba)

print(f"Training Log Loss: {train_log_loss:.4f}")
print(f"Validation Log Loss: {val_log_loss:.4f}")

In [ ]:
# Classification report
print("\nValidation Classification Report:")
print(classification_report(y_val, y_val_pred, 
                          target_names=['Model A Wins', 'Model B Wins', 'Tie']))

In [ ]:
test_columns_to_drop = ['id', 'prompt', 'response_a', 'response_b']
X_test = test.drop(columns=test_columns_to_drop)

print(f"Test feature matrix shape: {X_test.shape}")
print(f"Test features: {list(X_test.columns)}")


In [ ]:
# What's missing in the test data?
train_features = set(X.columns)
test_features = set(X_test.columns)
missing_features = train_features - test_features
extra_features = test_features - train_features

print(f"\nMissing features in test data: {missing_features}")
print(f"Extra features in test data: {extra_features}")

In [ ]:
# Handle missing by adding missing features to test data with default values (0 for encoded features)
for feature in missing_features:
    if 'encoded' in feature:
        # For encoded model features, use 0 (unknown model)
        X_test[feature] = 0
    else:
        # For other missing features, use 0 or appropriate default
        X_test[feature] = 0

# Ensure test data has the same columns in the same order as training data
X_test = X_test[X.columns]

print(f"\nTest data after handling missing features:")
print(f"Shape: {X_test.shape}")
print(f"Features match training: {list(X_test.columns) == list(X.columns)}")


In [ ]:
# Scale test data using the same scaler
X_test_scaled = scaler.transform(X_test)

# Make predictions on test set
test_predictions = model.predict_proba(X_test_scaled)

print(f"\nTest predictions shape: {test_predictions.shape}")

# Create submission DataFrame
submission = pd.DataFrame({
    'id': test['id'],
    'winner_model_a': test_predictions[:, 0],  # Probability for class 0
    'winner_model_b': test_predictions[:, 1],  # Probability for class 1
    'winner_tie': test_predictions[:, 2]       # Probability for class 2
})

# Display first few rows
print("\nSubmission file preview:")
print(submission.head())

In [ ]:
# Performing cross-validation
cv_scores = cross_val_score(model, X_train_scaled, y_train, 
                           cv=5, scoring='neg_log_loss')
cv_log_loss = -cv_scores.mean()

print(f"Cross-validation Log Loss: {cv_log_loss:.4f} (+/- {cv_scores.std() * 2:.4f})")

In [ ]:
# Scale test data using the same scaler
X_test_scaled = scaler.transform(X_test)

# Then Make predictions on the test set
test_predictions = model.predict_proba(X_test_scaled)

print(f"Test predictions shape: {test_predictions.shape}")
print("Ready for submission!")

In [ ]:
# Save submission file
submission.to_csv('submission.csv', index=False)
print(f"\nSubmission file saved as 'submission.csv'")
print(f"File contains {len(submission)} predictions")

# Verify the format matches the required format
print("\nVerification - First 3 rows:")
print(submission.head(3).to_string(index=False))